In [2]:
import sqlite3
import json

# SQLite 데이터베이스에 연결
conn = sqlite3.connect('ollama.db')
cursor = conn.cursor()

# 새로운 테이블 생성 (필요한 경우)
cursor.execute('''
CREATE TABLE IF NOT EXISTS messages (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    role TEXT,
    content TEXT
)
''')

# 기존 테이블에서 JSON 데이터를 읽어오는 쿼리
cursor.execute("SELECT message FROM message_store")

# 모든 행을 가져오기
rows = cursor.fetchall()

# 각 행의 JSON 데이터를 파싱하고 새로운 테이블에 삽입
for row in rows:
    json_data = row[0]
    parsed_data = json.loads(json_data)
    
    # 필요한 필드 추출
    role = parsed_data['data']['type']
    content = parsed_data['data']['content']
    
    # 새로운 테이블에 데이터 삽입
    cursor.execute('''
    INSERT INTO messages (role, content)
    VALUES (?, ?)
    ''', (role, content))

# 변경사항 저장
conn.commit()

# 연결 닫기
conn.close()

In [3]:
import sqlite3

# SQLite 데이터베이스에 연결
conn = sqlite3.connect('ollama.db')
cursor = conn.cursor()

# role 컬럼의 값을 변경하는 쿼리
cursor.execute('''
UPDATE messages
SET role = CASE
    WHEN role = 'human' THEN 'user'
    WHEN role = 'ai' THEN 'assistant'
    ELSE role
END
''')

# 변경사항 저장
conn.commit()

# 연결 닫기
conn.close()